<a href="https://colab.research.google.com/github/Upayukta/ThingsToDo/blob/main/CRS_CHECK_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# Install required packages including ipywidgets
!pip install -q laspy[laszip] pyproj pandas openpyxl ipywidgets

import os
import io
import re
from google.colab import drive, auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import pandas as pd
from pyproj import CRS
from laspy import LazBackend
import laspy
import ipywidgets as widgets
from IPython.display import display, clear_output

# Mount Google Drive & Authenticate API
drive.mount('/content/drive')
auth.authenticate_user()
credentials, project = google.auth.default()
drive_service = build('drive', 'v3', credentials=credentials)

# Create UI Widgets for Link Input
link_input = widgets.Text(
    value='',
    placeholder='Paste your Google Drive folder link here...',
    description='Drive Link:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

submit_button = widgets.Button(
    description='SUBMIT',
    button_style='success',
    icon='check'
)

output_area = widgets.Output()

def extract_folder_id(link):
    match = re.search(r"/folders/([a-zA-Z0-9_-]+)", link)
    if not match:
        raise ValueError(f"Invalid Google Drive folder link: {link}")
    return match.group(1)

def on_submit_clicked(b):
    global FOLDER_ID
    with output_area:
        clear_output()
        folder_link = link_input.value.strip()
        try:
            if not folder_link:
                print("⚠️ Please paste a valid folder link first.")
                return
            FOLDER_ID = extract_folder_id(folder_link)
            print("✅ Folder ID captured successfully! You can now run Cell 2.")
        except Exception as e:
            print(f"❌ Error: {str(e)}")

submit_button.on_click(on_submit_clicked)

# Display the interactive elements
display(widgets.VBox([link_input, submit_button, output_area]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -*- coding: utf-8 -*-
laz_files = []
folder_cache = {}

def get_folder_name(folder_id):
    if folder_id in folder_cache:
        return folder_cache[folder_id]
    info = drive_service.files().get(
        fileId=folder_id,
        fields="id,name",
        supportsAllDrives=True
    ).execute()
    folder_cache[folder_id] = info["name"]
    return info["name"]

def scan_laz_files(folder_id, parent_path=""):
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            pageSize=1000,
            pageToken=page_token,
            fields="nextPageToken,files(id,name,mimeType,size,modifiedTime,parents)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()

        for item in response.get("files", []):
            name = item["name"]
            current_path = parent_path + "/" + name

            if item["mimeType"] == "application/vnd.google-apps.folder":
                scan_laz_files(item["id"], current_path)
            elif name.lower().endswith(".laz"):
                laz_files.append({
                    "File Name": name,
                    "Drive File ID": item["id"],
                    "Google Drive Path": current_path,
                    "File Size (MB)": round(int(item.get("size", 0)) / (1024**2), 3),
                    "Last Modified": item.get("modifiedTime", "")
                })

        page_token = response.get("nextPageToken")
        if not page_token:
            break

root_name = get_folder_name(FOLDER_ID)
scan_laz_files(FOLDER_ID, root_name)

print("LAZ SCAN COMPLETED")
print("-------------------")
print(f"Total LAZ files found: {len(laz_files)}")

LAZ SCAN COMPLETED
-------------------
Total LAZ files found: 17


In [ ]:
# -*- coding: utf-8 -*-
def download_laz_to_memory(file_id):
    request = drive_service.files().get_media(
        fileId=file_id,
        supportsAllDrives=True
    )
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    return buffer

non_utm_files = []

for file_info in laz_files:
    print(f"Checking: {file_info['File Name']}")
    try:
        laz_buffer = download_laz_to_memory(file_info["Drive File ID"])

        with laspy.open(laz_buffer, laz_backend=LazBackend.Laszip) as reader:
            header = reader.header
            crs = header.parse_crs()

            is_utm = False
            crs_desc = "Missing or Unknown CRS"

            if crs:
                proj_crs = CRS.from_user_input(crs)
                crs_desc = proj_crs.name

                if proj_crs.is_projected and "UTM" in proj_crs.name.upper():
                    is_utm = True

            if not is_utm:
                print(f"  [!] Non-UTM detected: {crs_desc}")
                non_utm_files.append({
                    "File Name": file_info["File Name"],
                    "Detected CRS": crs_desc
                })
            else:
                print(f"  [✓] UTM Projection confirmed.")

    except Exception as e:
        print(f"  [X] Error reading file: {str(e)}")
        non_utm_files.append({
            "File Name": file_info["File Name"],
            "Detected CRS": f"Error: {str(e)}"
        })

print("\n==============================")
print(f"REPORT: Total non-UTM files found: {len(non_utm_files)}")
print("==============================")
for item in non_utm_files:
    print(f"File: {item['File Name']} | CRS: {item['Detected CRS']}")

Checking: CD-GCP-E102-1_WITHOUT GCP N.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E102-1_WITHOUT GCP - DOUBLE LAYER.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E116-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E116-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E114-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E114-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E112-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E112-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E110-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E110-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E108-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E108-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E106-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E106-1.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E104-2.laz
  [✓] UTM Projection confirmed.
Checking: CD-GCP-E104-1.laz
  [✓] UTM Projection confirmed.

In [2]:
!pip freeze > requirements.txt